# Accelerator Orbit Alignment Hackathon: Beam Orbit Benchmark
## Solution & Reference Benchmark Edition (with Weights & Biases / W&B Leaderboard Integration)

> **Worked solution.** TODO 1 uses an accurate MLP surrogate, and Tracks 1–3 share one strategy: estimate the orbit distortion from `info["currents"]`, solve for the currents that cancel it, and move there in equal steps of at most 0.8 A. Expected scores: Baseline 0 635.277, Track 1 924.192, Track 2 939.981, Track 3 909.942. The unmodified participant notebook scores 779.169 / 590.559 / 878.384 on Tracks 1–3.

**Workshop & Challenge Context:**
* **Author:** Chong Shik Park, Ph.D.
* **Affiliation:** Korea University, Sejong
* **Date:** November 19, 2026
* **Target Audience:** Graduate Students & Early-Career Accelerator Researchers
* **Curricular Alignment:** 
  * **Lecture 1:** Chong Shik Park — AI/ML Foundations and Closed-Loop Control for Particle Accelerators
  * **Lecture 2:** Thorsten Hellert — Agentic AI, Autonomous Operations & Scientific Workflows

---

### 🎯 Challenge Mission
In modern particle accelerators and beam transport lines, beam alignment is critical to ensure high transmission efficiency and prevent beam loss. 
In this hackathon, you will develop closed-loop control algorithms to steer the beam through a **15-meter transport line** with:
* **6 Dipole Steering Magnets:** $X = [\mathrm{HCM1}, \mathrm{VCM1}, \mathrm{HCM2}, \mathrm{VCM2}, \mathrm{HCM3}, \mathrm{VCM3}]^T \in [-5.0\text{ A}, +5.0\text{ A}]$
* **12 Beam Position Monitors (BPMs):** $Y \in \mathbb{R}^{12}$, measuring transverse beam offsets in millimeters (mm).
* **Non-Linear Iron Saturation:** $B(I) \approx I - 0.015 \cdot I^3$, causing progressive softening at higher corrector currents.
* **Objective:** Align the beam so that the Root-Mean-Square (RMS) displacement is below **$0.030\text{ mm}$** within a budget of **50 control steps**:
  $$\text{RMS} = \sqrt{\frac{1}{12}\sum_{i=1}^{12} Y_i^2} < 0.030\text{ mm}$$

---

### 🏆 Three Parallel Competition Tracks:
1. **Track 1 (Supervised Digital Twin Inversion):** Train a non-linear neural surrogate model (MLP) on offline tracking data, and use an optimization solver to invert the surrogate for optimal corrective kicks.
2. **Track 2 (Closed-Loop Reinforcement Learning / Adaptive Policy):** Design an interactive feedback policy agent that continuously senses BPM offsets and outputs delta current kicks $\Delta I_c$ with physics-informed priors and adaptive damping.
3. **Track 3 (Bayesian Optimization Controller):** Build a sample-efficient Gaussian Process controller with Lower Confidence Bound (LCB) acquisition to actively align the beam under non-linear saturation.


In [1]:
# Step 0: Imports, Environment Setup, and Weights & Biases (W&B) Configuration
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

# Import standardized simulation, controllers, benchmark & visualization from src/
from src import (
    AcceleratorOrbitGymEnv,
    ClassicalSVDController,
    SupervisedMLPController,
    RLOrbitChallengeAgent,
    BayesianOptimizationController,
    generate_offline_dataset,
    evaluate_hackathon_benchmark,
    print_benchmark_leaderboard,
    draw_beamline,
    plot_benchmark_trajectories
)

# ------------------------------------------------------------------------------
# Hackathon Configuration (Customize for Your Team!)
# ------------------------------------------------------------------------------
# >>> [PARTICIPANT MODIFY HERE]: Change to your registered team name!
TEAM_NAME = "Team_ReferenceSolution"

# >>> [DO NOT MODIFY]: Official workshop competition settings
WANDB_PROJECT = "ai4acc-mini-hackathon-2026" # [DO NOT CHANGE] Official project name for live leaderboard
WANDB_ENTITY = "ai4acc-mini-hackathon-2026"  # [DO NOT CHANGE] Preset workshop entity
USE_WANDB = False                            # Offline local evaluation mode for solution notebook

print(f"Setup complete! Ready to compete as: {TEAM_NAME}")


Setup complete! Ready to compete as: Team_ReferenceSolution


## 1. Accelerator Lattice Physics & 2D Schematic

The challenge beamline section consists of:
* **Actuators:** 6 dipole steering magnets (3 Horizontal `HCM1-3`, 3 Vertical `VCM1-3`).
* **Optics:** Alternating quadrupole magnets (`QF` focusing, `QD` defocusing) shaping betatron oscillations.
* **Diagnostics:** 12 Beam Position Monitors (`BPM1` to `BPM12`) spaced along the 15-meter transport axis.

The 2D schematic diagram below illustrates the arrangement of steering magnets, quadrupoles, and diagnostic monitors along the longitudinal beam axis $s$.


In [2]:
# Step 1: Display 2D Accelerator Beamline Schematic Diagram
# [DO NOT MODIFY - Diagnostic Lattice Visualization]
fig = draw_beamline(save_path="img/beamline.png", show=True)


Lattice schematic saved to ../img/beamline.png


/home/cspark/Work/projects/ai4acc-mini-hackathon/src/visualization.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Standard Gymnasium Environment (`AcceleratorOrbitGymEnv`)

The accelerator simulator is encapsulated in a standardized **Gymnasium** environment (`src/environment.py`):
* **Observation Space:** $\mathbf{Y} \in \mathbb{R}^{12}$, bounded in $[-10.0\text{ mm}, +10.0\text{ mm}]$ (12 BPM readings).
* **Action Space:** $\Delta \mathbf{I} \in \mathbb{R}^6$, bounded in $[-0.8\text{ A}, +0.8\text{ A}]$ per step.
* **Hardware Limit:** Total current state $\mathbf{I}_{t+1} = \text{clip}(\mathbf{I}_t + \Delta \mathbf{I}, -5.0, +5.0)\text{ A}$.
* **Physical Response Model:**
  $$\mathbf{Y} = \mathbf{I} \cdot \mathbf{R} - 0.015 \cdot (\mathbf{I}^3 \cdot \mathbf{R}) + \mathbf{Y}_0 + \boldsymbol{\epsilon}$$
  where $\mathbf{R}$ is the $6 \times 12$ linear transfer matrix, $-0.015 \mathbf{I}^3 \mathbf{R}$ represents soft iron core saturation, $\mathbf{Y}_0 = \mathbf{a} \cdot \mathbf{R}$ is the initial distortion from unknown kicks $a_k \sim \mathcal{U}(-2.5, 2.5)$ A at all six corrector locations (drawn per episode seed; cancelling it needs currents up to about 2.8 A), and $\boldsymbol{\epsilon} \sim \mathcal{N}(0, 0.010^2\text{ mm}^2)$ is Gaussian sensor noise.
* **Customizable Reward & Penalty Structure (for RL & Policy Optimization):**
  $$r_t = - w_{\text{rms}} \cdot \text{RMS}_{t+1} - w_{\text{act}} \cdot \|\Delta \mathbf{I}\|_2^2 - c_{\text{step}} + (B_{\text{success}} \text{ if terminated else } 0.0)$$
  * $w_{\text{rms}}$ (`rms_weight`, default `10.0`): Penalizes beam displacement from center axis.
  * $w_{\text{act}}$ (`action_weight`, default `0.01`): Penalizes large actuator kick effort to prevent current saturation.
  * $c_{\text{step}}$ (`step_penalty`, default `0.0`): Step cost deducted each turn. **Increasing this (e.g. 0.5 to 1.5) incentivizes RL agents to converge in fewer steps!**
  * $B_{\text{success}}$ (`success_bonus`, default `50.0`): Completion reward when $\text{RMS} < 0.030\text{ mm}$.
  * `reward_fn`: Optional custom callable `(rms, action, terminated, truncated, step) -> float` for advanced reward shaping.
* **Termination:** Episode succeeds when $\text{RMS} < 0.030\text{ mm}$, or truncates after `max_steps = 50` steps.


In [3]:
# Step 2: Instantiate Standardized Hackathon Gymnasium Environment
# ==============================================================================
# [OPTIONAL TODO / CAN MODIFY FOR RL POLICY TRAINING & OPTIMIZATION]
# ==============================================================================
# You can customize reward/penalty weights or pass a custom reward_fn callback:
env = AcceleratorOrbitGymEnv(
    rms_weight=10.0,      # [CAN MODIFY] Penalty weight for orbit RMS error
    action_weight=0.01,   # [CAN MODIFY] Penalty weight for corrector kick effort
    step_penalty=0.5,     # [CAN MODIFY] Step penalty (env default 0.0) to incentivize faster convergence!
    success_bonus=50.0    # [CAN MODIFY] Completion bonus upon reaching RMS < 0.030 mm
    # reward_fn=None      # [CAN MODIFY] Optional: pass custom_reward_fn(rms, action, term, trunc, step)
)

# [DO NOT MODIFY]: Reset to canonical test seed 42 to inspect initial uncorrected orbit
obs, info = env.reset(seed=42)

print("=" * 68)
print("   Hackathon Gymnasium Environment Instantiated Successfully!")
print("=" * 68)
print(f"  • Observation Vector Shape : {obs.shape} (12 BPM stations in mm)")
print(f"  • Action Vector Shape      : {env.action_space.shape} (6 Corrector kick deltas in A)")
print(f"  • Action Limits (per step) : [{env.action_space.low[0]:.1f} A, +{env.action_space.high[0]:.1f} A]")
print(f"  • Hardware Current Bounds  : [-5.0 A, +5.0 A]")
print(f"  • Target RMS Tolerance     : < {env.target_rms:.3f} mm")
print(f"  • Max Control Budget       : {env.max_steps} steps per episode")
print(f"  • Initial Uncorrected RMS  : {info['rms']:.4f} mm")
print(f"  • Reward Shaping Params    : rms_w={env.rms_weight}, act_w={env.action_weight}, step_cost={env.step_penalty}, bonus={env.success_bonus}")
print("=" * 68)


   Hackathon Gymnasium Environment Instantiated Successfully!
  • Observation Vector Shape : (12,) (12 BPM stations in mm)
  • Action Vector Shape      : (6,) (6 Corrector kick deltas in A)
  • Action Limits (per step) : [-0.8 A, +0.8 A]
  • Hardware Current Bounds  : [-5.0 A, +5.0 A]
  • Target RMS Tolerance     : < 0.030 mm
  • Max Control Budget       : 50 steps per episode
  • Initial Uncorrected RMS  : 1.6246 mm
  • Reward Shaping Params    : rms_w=10.0, act_w=0.01, step_cost=0.5, bonus=50.0


## 3. Supervised Forward Modeling: Linear Model vs. Neural Surrogate

In accelerator operations, numerical tracking simulation datasets can be converted into fast machine learning digital twins (Lecture 1).
Here, we collect 1,500 simulation shots (80% train, 20% test) across currents $[-3.0\text{ A}, +3.0\text{ A}]$ (the correction needs up to about 2.8 A) and train:
1. **A Linear Ridge Model:** Serves as a baseline, but cannot account for non-linear core saturation.
2. **A Multi-Layer Perceptron (MLP):** A deep neural network capable of fitting the non-linear cubic saturation curve.


In [4]:
# Step 3: Offline Data Generation & Surrogate Training
# [OPTIONAL CAN MODIFY]: You can increase n_samples (e.g. 2000, 3000) or adjust sampling range:
X_offline, Y_offline = generate_offline_dataset(
    env,
    n_samples=3000,               # [SOLUTION] 3,000 samples (default 1,500)
    current_range=(-3.0, 3.0),    # [CAN MODIFY] Current sampling range in Amperes
    noise_std=0.010,              # [DO NOT MODIFY] Physical sensor noise floor (0.010 mm)
    random_state=42
)

# [DO NOT MODIFY]: Standard 80/20 train/test evaluation split
X_train, X_test, Y_train, Y_test = train_test_split(X_offline, Y_offline, test_size=0.20, random_state=42)
print(f"Dataset generated: X_train shape={X_train.shape}, Y_train shape={Y_train.shape}")

# ==============================================================================
# TODO 1: Train Linear Baseline & Neural Surrogate Model
# ==============================================================================
# INSTRUCTIONS FOR PARTICIPANTS:
# Train two surrogate models to predict the 12 BPM readings from 6 corrector currents:
#   1. Linear Model (Ridge Regression): Fast baseline, but linear only.
#   2. Multi-Layer Perceptron (MLPRegressor): Deep feedforward neural network that
#      learns the cubic iron core magnetic saturation: B(I) ~ I - 0.015 * I^3.
#
# ------------------------------------------------------------------------------
# EXAMPLES FOR NEWBIE PARTICIPANTS:
#
# Example 1: Train Ridge Linear Regression
#   linear_model = Ridge(alpha=1.0)
#   linear_model.fit(X_train, Y_train)
#
# Example 2: Train Multi-Layer Perceptron (MLPRegressor)
#   surrogate_model = MLPRegressor(
#       hidden_layer_sizes=(128, 64, 32), # 3 hidden layers: 128 -> 64 -> 32 neurons
#       activation='relu',               # Activation function: f(x) = max(0, x)
#       solver='adam',                   # Adam gradient-based optimizer
#       alpha=1e-4,                      # L2 regularization penalty
#       learning_rate_init=0.001,        # Initial learning rate
#       max_iter=400,                    # Training epochs
#       random_state=42
#   )
#   surrogate_model.fit(X_train, Y_train)
#
# ------------------------------------------------------------------------------
# 💡 HINTS TO GET HIGHER SCORES:
#   • Network Architecture: Try wider or deeper layers like (128, 128), (256, 128, 64).
#     A more expressive surrogate provides a more accurate gradient for the optimizer in Track 1!
#   • Training Sample Size: Increasing n_samples (e.g. 2000 or 3000) improves test generalization.
#   • Regularization: Tune alpha (e.g., 1e-4 vs 1e-5) to avoid overfitting the sensor noise.
#   • Accuracy Benchmark: Target test RMSE is < 0.015 mm (near the 0.010 mm sensor noise floor).
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Train Linear Baseline
# ------------------------------------------------------------------------------
# >>> [OPTIONAL CAN MODIFY]: Tune alpha regularization parameter:
linear_model = Ridge(alpha=1.0).fit(X_train, Y_train)

# [DO NOT MODIFY]: Test set RMSE evaluation
rmse_linear = root_mean_squared_error(Y_test, linear_model.predict(X_test))

# ------------------------------------------------------------------------------
# 2. Train Multi-Layer Perceptron (MLP) Neural Surrogate
# ------------------------------------------------------------------------------
# >>> [PARTICIPANT MODIFY HERE - TODO 1] <<<
# Tune hidden_layer_sizes, activation, alpha, learning_rate_init, max_iter to minimize test RMSE:
# [SOLUTION]: a small smooth (tanh) network trained with the full-batch L-BFGS solver fits the
# cubic saturation far better than the default ReLU/Adam setup, which stops after ~85 iterations
# with ~0.11 mm test RMSE. This reaches ~0.012 mm, close to the 0.010 mm BPM noise floor.
surrogate_model = MLPRegressor(
    hidden_layer_sizes=(64, 64),      # [SOLUTION] default (128, 64, 32)
    activation='tanh',                # [SOLUTION] default 'relu'
    solver='lbfgs',                   # [SOLUTION] default 'adam'
    alpha=1e-5,                       # [SOLUTION] default 1e-4
    max_iter=1500,                    # [SOLUTION] default 400
    random_state=42
).fit(X_train, Y_train)

# [DO NOT MODIFY]: Test set RMSE evaluation
rmse_mlp = root_mean_squared_error(Y_test, surrogate_model.predict(X_test))

print("=" * 68)
print("SURROGATE MODEL PREDICTIVE ACCURACY (TEST SET):")
print("=" * 68)
print(f"  • Linear Ridge Test RMSE   : {rmse_linear:.4f} mm")
print(f"  • Neural MLP Test RMSE     : {rmse_mlp:.4f} mm (Captures cubic core saturation)")
print(f"  • Gaussian BPM Noise Floor : {env.noise_std:.4f} mm")
print("=" * 68)


Dataset generated: X_train shape=(2400, 6), Y_train shape=(2400, 12)


SURROGATE MODEL PREDICTIVE ACCURACY (TEST SET):
  • Linear Ridge Test RMSE   : 0.0868 mm
  • Neural MLP Test RMSE     : 0.0115 mm (Captures cubic core saturation)
  • Gaussian BPM Noise Floor : 0.0100 mm


---
## 4. Benchmark Controllers: Classical, Supervised, RL, and Bayesian Optimization Agents

Every competitor controller must implement the standardized agent interface:
```python
def act(self, obs: np.ndarray, info: Optional[dict] = None) -> np.ndarray
```
where `obs` has shape `(12,)` and the returned action is an array of shape `(6,)` in $[-0.8\text{ A}, +0.8\text{ A}]$.

1. **Baseline 0 (Classical SVD Controller):** Inverts the linear Orbit Response Matrix via Moore-Penrose pseudo-inverse: $\Delta \mathbf{I} = - \alpha (\mathbf{R}^T)^{\dagger} \mathbf{Y}$.
2. **Baseline 1 / Track 1 (Supervised Neural Inversion):** Solves an optimization problem per step using the trained neural surrogate digital twin:
   $$\min_{\Delta \mathbf{I}} \|\mathbf{Y} + f_{\theta}(\Delta \mathbf{I})\|^2 + \lambda \|\Delta \mathbf{I}\|^2$$
3. **Challenge Track 2 (Closed-Loop RL / Adaptive Agent):** Combines physics prior feedback with adaptive step damping:
   $$\Delta \mathbf{I}_t = \text{damping}(t) \cdot \left[ - \alpha (\mathbf{R}^T)^{\dagger} \mathbf{Y} + \mathbf{W}_{\text{tune}} \mathbf{Y} + \mathbf{b} \right]$$
4. **Challenge Track 3 (Bayesian Optimization Controller):** Uses a Gaussian Process surrogate with epistemic uncertainty estimation to evaluate an acquisition function (Lower Confidence Bound, LCB) balancing exploitation of predicted orbit corrections with exploration:
   $$\alpha_{\text{LCB}}(\Delta \mathbf{I}) = \text{RMS}(\mathbf{Y}_{\text{obs}} + \mu_{\text{GP}}(\Delta \mathbf{I})) - \kappa \cdot \sigma_{\text{GP}}(\Delta \mathbf{I}) + \lambda \|\Delta \mathbf{I}\|^2$$


In [5]:
# Step 4: Define Benchmark Controllers -- WORKED SOLUTION
# ==============================================================================
# Solution approach shared by Tracks 1-3 (model inversion on the TOTAL corrector current):
#   1. info["currents"] gives the total current I applied so far. With a model f(I) of the
#      orbit response, the fixed orbit distortion is estimated as  d = obs - f(I).
#   2. d is averaged over the steps of the episode, which reduces BPM noise.
#   3. The target currents I* minimize ||d + f(I*)||^2: the orbit they produce cancels d.
#   4. The kick I* - I is split into equal steps of at most 0.8 A. Splitting (instead of
#      clipping) keeps the intermediate orbit well above 0.030 mm, so the episode does not
#      stop early on a barely-converged, low-precision reading.
# The tracks differ only in the model f: an MLP (Track 1), the exact physics with cubic
# saturation (Track 2), or a Gaussian process with uncertainty (Track 3).
# ==============================================================================
import scipy.optimize as opt
from sklearn.gaussian_process.kernels import ConstantKernel


class ModelInversionController:
    """Estimate the orbit distortion from the measured orbit and the model's prediction for the
    present currents, solve for the currents that cancel it, and move there in equal steps."""
    max_step = 0.8

    def __init__(self, forward, bounds=5.0):
        self.forward = forward          # f(I): (6,) total currents -> (12,) orbit response
        self.bounds = bounds
        self._drift_sum, self._n = None, 0

    def solve_target(self, drift, I_now):
        def objective(I):
            r = drift + self.forward(I)
            return float(r @ r)
        res = opt.minimize(objective, I_now, method="L-BFGS-B", bounds=[(-self.bounds, self.bounds)] * 6)
        return res.x

    def act(self, obs, info=None):
        info = info or {}
        I_now = np.asarray(info.get("currents", np.zeros(6)), dtype=float)
        if info.get("step", 0) == 0:                     # new episode: reset the distortion estimate
            self._drift_sum, self._n = np.zeros(len(obs)), 0
        self._drift_sum += obs - self.forward(I_now)
        self._n += 1
        drift = self._drift_sum / self._n                # averaged over the episode
        delta = self.solve_target(drift, I_now) - I_now
        n_moves = max(1, int(np.ceil(np.max(np.abs(delta)) / self.max_step - 1e-9)))
        return np.clip(delta / n_moves, -self.max_step, self.max_step)


class PhysicsInversionAgent(ModelInversionController):
    """Track 2: exact physics B(I) = I - 0.015 I^3, solved analytically (pseudo-inverse + Newton)."""
    def __init__(self, R_matrix):
        self.R_pinv = np.linalg.pinv(R_matrix.T)
        super().__init__(lambda I: (I - 0.015 * I**3) @ R_matrix)

    def solve_target(self, drift, I_now):
        b = -self.R_pinv @ drift                         # magnetic field that cancels the distortion
        I = b.copy()
        for _ in range(8):                               # Newton iterations on I - 0.015 I^3 = b
            I -= (I - 0.015 * I**3 - b) / (1.0 - 0.045 * I**2)
        return np.clip(I, -self.bounds, self.bounds)


class GPLCBController(ModelInversionController):
    """Track 3: GP posterior mean as the model; with kappa > 0 the LCB acquisition also rewards
    exploring uncertain regions. kappa = 0 (pure exploitation) scored best here."""
    def __init__(self, gp_model, kappa=0.0):
        self.gp, self.kappa = gp_model, float(kappa)
        super().__init__(lambda I: gp_model.predict(I.reshape(1, -1))[0])

    def solve_target(self, drift, I_now):
        if self.kappa == 0.0:
            return super().solve_target(drift, I_now)
        def acquisition_lcb(I):
            mu, sd = self.gp.predict(I.reshape(1, -1), return_std=True)
            return float(np.sqrt(np.mean((drift + mu[0])**2)) - self.kappa * np.mean(sd))
        return opt.minimize(acquisition_lcb, I_now, method="L-BFGS-B", bounds=[(-self.bounds, self.bounds)] * 6).x


# ------------------------------------------------------------------------------
# 1. Baseline 0: Classical Linear ORM Pseudo-Inverse Controller (fixed reference)
# ------------------------------------------------------------------------------
b0 = ClassicalSVDController(env.R, gain=0.45)

# ------------------------------------------------------------------------------
# 2. Baseline 1 (Track 1): invert the accurate MLP surrogate from TODO 1
# ------------------------------------------------------------------------------
b1 = ModelInversionController(lambda I: surrogate_model.predict(I.reshape(1, -1))[0])

# ------------------------------------------------------------------------------
# 3. Track 2: closed-loop agent with the exact physics model
# ------------------------------------------------------------------------------
b2 = PhysicsInversionAgent(env.R)

# ------------------------------------------------------------------------------
# 4. Track 3: Bayesian optimization with a Gaussian process
# ------------------------------------------------------------------------------
# 700 calibration shots in [-3.5, 3.5] A cover the currents needed for correction (up to about 3 A);
# an ARD Matern-5/2 kernel (one length scale per corrector) with a fixed BPM-noise term fits them well.
X_gp, Y_gp = generate_offline_dataset(env, n_samples=700, current_range=(-3.5, 3.5), noise_std=0.010, random_state=7)
gp_kernel = ConstantKernel(1.0) * Matern(length_scale=[2.0] * 6, nu=2.5) + WhiteKernel(noise_level=1e-4, noise_level_bounds='fixed')
gp_surrogate = GaussianProcessRegressor(kernel=gp_kernel, normalize_y=True, random_state=42)
gp_surrogate.fit(X_gp, Y_gp)
b3 = GPLCBController(gp_surrogate, kappa=0.0)

# ------------------------------------------------------------------------------
# Optional callbacks for Baseline 1, Track 2 and Track 3 (see Example D in the participant notebook)
# ------------------------------------------------------------------------------
# The solution needs no callbacks: each controller reads info["currents"] and info["step"] directly.
# Baseline 0 is the fixed reference and always runs without callbacks.
CALLBACKS = {
    "b1": {"step_callback": None, "episode_callback": None},  # Baseline 1 / Track 1
    "b2": {"step_callback": None, "episode_callback": None},  # Track 2
    "b3": {"step_callback": None, "episode_callback": None},  # Track 3
}

print("All four competitor controllers (Baseline 0, Track 1, Track 2, Track 3) initialized successfully.")

All four competitor controllers (Baseline 0, Track 1, Track 2, Track 3) initialized successfully.


---
## 5. Standardized Benchmark Evaluation & Live W&B Scoring

To ensure fair and rigorous comparison, agents are evaluated across **30 standardized evaluation seeds** (`seeds = range(1, 31)`) featuring diverse physical drift perturbations.

### 🎯 Official Multi-Objective Evaluation Criteria
The benchmark evaluation engine (`src/benchmark.py`) scores **each seed from 0 to 1000 points**, and your **Hackathon Benchmark Score** is the mean over the 30 seeds. A seed earns points on three dimensions:
1. **Alignment Success (400 pts):** Did the agent achieve $\text{RMS} < 0.030\text{ mm}$ within the 50-step episode limit? A failed seed scores **0**, so robustness comes first.
2. **Convergence Speed (up to 300 pts):** Full points for converging in 1 step, falling linearly to 0 at 20 steps (episodes may run up to 50 steps).
3. **Residual Precision (up to 300 pts):** Landing below the $0.030\text{ mm}$ threshold earns more the closer you get to the $0.010\text{ mm}$ sensor noise floor (log scale).

Corrector current norm $\overline{\|\mathbf{I}\|}$ is still reported, but it is not part of the score. Reference controllers score between about 590 and 880 points.

---

### 💡 Strategic Tuning Cheat-Sheet: How to Beat the Baseline Leaderboard!

| Tuning Target | What to Modify | Impact on Performance & Score |
| :--- | :--- | :--- |
| **Faster Convergence** | Feedback gain $\alpha$ up to $\approx 1.2$ in `act()` | Gain 0.70 cuts mean steps from 9.4 to about 6; gain 1.2 to about 3.7. The 0.8 A step limit keeps most seeds at 3 or more steps. |
| **Adaptive Gain Schedule** | Dynamic gain based on current RMS in `act()` | Aggressive kicks when far (`> 0.15 mm`); fine-tuned kicks when close (`< 0.05 mm`). |
| **Precision RMS Bonus** | Damping cooling factor $\gamma \in [0.015, 0.035]$ | Prevents terminal oscillations, allowing the beam to land near the $0.010\text{ mm}$ noise floor. |
| **Non-Linear Boost** | Invert $B(\mathbf{I}) = \mathbf{I} - 0.015\,\mathbf{I}^3$ on the total current from `info[\"currents\"]` (TODO 2 hint 4) | Cancels cubic iron saturation (up to 12% at 2.8 A), so a gain near 1.0 lands precisely in about 3.7 steps (924 points). |
| **Reward / Penalty Shaping** | Increase `step_penalty` ($0.5$–$1.5$) & `action_weight` in `env` | In RL training, penalizing steps forces 3–4 step solutions; penalizing kicks minimizes current norm! |
| **Precision Reward Shaping** | Add custom `reward_fn` with bonus when $\text{RMS} < 0.020\text{ mm}$ | Trains RL policy to aim for the $0.010\text{ mm}$ sensor noise floor instead of stopping at $0.029\text{ mm}$. |
| **Track 1 Optimizer Fix** | Bound search to `[(-0.8, 0.8)]` & warm-start | Prevents action clipping and boosts Track 1 success rate dramatically! |

When `USE_WANDB = True`, telemetry, seed breakdowns, and scores are automatically uploaded to **Weights & Biases (W&B)**.


In [6]:
# Step 5: Standardized Evaluation Harness with Weights & Biases Logging
# ==============================================================================
# [DO NOT MODIFY - Standardized 30-Seed Evaluation Protocol]
# ==============================================================================
# Evaluates each competitor controller across 30 standardized seeds (range 1 to 31).
# Callbacks registered in CALLBACKS (Step 4) are passed to Baseline 1, Track 2 and Track 3 only.
# Telemetry, seed breakdowns, and scores are automatically uploaded to W&B.
print("Running Official Hackathon Benchmark across 30 Test Seeds...")

res_b0 = evaluate_hackathon_benchmark(env, b0, agent_name="Baseline0_LinearSVD", team_name=TEAM_NAME, use_wandb=USE_WANDB, project=WANDB_PROJECT, entity=WANDB_ENTITY)
res_b1 = evaluate_hackathon_benchmark(env, b1, agent_name="Baseline1_SupervisedMLP", team_name=TEAM_NAME, use_wandb=USE_WANDB, project=WANDB_PROJECT, entity=WANDB_ENTITY, **CALLBACKS["b1"])
res_b2 = evaluate_hackathon_benchmark(env, b2, agent_name="Track2_RLAgent", team_name=TEAM_NAME, use_wandb=USE_WANDB, project=WANDB_PROJECT, entity=WANDB_ENTITY, **CALLBACKS["b2"])
res_b3 = evaluate_hackathon_benchmark(env, b3, agent_name="Track3_BayesianOpt", team_name=TEAM_NAME, use_wandb=USE_WANDB, project=WANDB_PROJECT, entity=WANDB_ENTITY, **CALLBACKS["b3"])

# Display Official Leaderboard Table
print_benchmark_leaderboard([
    ("Track 3 (Bayesian Optimization Controller)", res_b3),
    ("Baseline 1 (Supervised MLP Inversion)", res_b1),
    ("Baseline 0 (Linear ORM SVD Controller)", res_b0),
    ("Track 2 (RL Closed-Loop Agent)", res_b2),
])


Running Official Hackathon Benchmark across 30 Test Seeds...


                            OFFICIAL HACKATHON BENCHMARK LEADERBOARD
Rank  | Agent / Controller Name                    | Success  | Steps  | Final RMS  |     Score
-----------------------------------------------------------------------------------------------
#1    | Track 2 (RL Closed-Loop Agent)             |   100.0% |    3.7 |   0.0101mm |   939.981
#2    | Baseline 1 (Supervised MLP Inversion)      |   100.0% |    3.7 |   0.0110mm |   924.192
#3    | Track 3 (Bayesian Optimization Controller) |   100.0% |    3.7 |   0.0118mm |   909.942
#4    | Baseline 0 (Linear ORM SVD Controller)     |   100.0% |    9.4 |   0.0235mm |   635.277


---
## 6. Visualizing Trajectory Alignment & Dynamic Convergence

In this step, we evaluate each controller on canonical test seed 42 to inspect:
1. **Beamline Trajectory Alignment (Subplot 1):** Uncorrected beam displacement vs. final corrected orbits across all 12 BPM stations.
2. **Convergence Dynamics (Subplot 2):** Orbit RMS error decay over successive interaction steps towards the $0.030\text{ mm}$ tolerance line.


In [7]:
# Step 6: Generate Dual Trajectory Overlay & Dynamic Convergence Curves
# ==============================================================================
# [DO NOT MODIFY - Canonical Diagnostic Visualization]
# ==============================================================================
fig = plot_benchmark_trajectories(
    env,
    controllers={
        "Linear SVD": b0,
        "Supervised MLP": b1,
        "RL Agent": b2,
        "Bayesian Opt": b3
    },
    seed=42,
    save_path="img/benchmark_trajectories_solution.png",  # separate file: the slides use the reference figure
    show=True
)


/home/cspark/Work/projects/ai4acc-mini-hackathon/src/visualization.py:162: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7. Final Team Showcase Cell (16:30–17:00)
Run this single summary cell during team presentations to display your official Hackathon submission score!


In [8]:
# Step 7: Final Team Showcase Presentation Summary
# ==============================================================================
# [DO NOT MODIFY - Automated Showcase Scorecard for Presentation]
# ==============================================================================
best_agent_name, best_metrics = max(
    [("Track 3 (Bayesian Optimization)", res_b3),
     ("Track 2 (RL Closed-Loop Agent)", res_b2),
     ("Baseline 1 (Supervised MLP)", res_b1),
     ("Baseline 0 (Linear SVD)", res_b0)],
    key=lambda item: item[1]["score"]
)

print("=" * 68)
print(f"   TEAM: {TEAM_NAME} — OFFICIAL HACKATHON BENCHMARK SUBMISSION")
print("=" * 68)
print(f"  • Best Performing Agent  : {best_agent_name}")
print(f"  • Official Total Score   : {best_metrics['score']:.3f} points")
print(f"  • Alignment Success Rate : {best_metrics['success_rate']:.1f}%")
print(f"  • Mean Steps to Target   : {best_metrics['mean_steps']:.1f} steps")
print(f"  • Mean Residual Orbit RMS: {best_metrics['mean_rms']:.4f} mm")
print(f"  • Mean Corrector Norm    : {best_metrics['mean_norm']:.2f} A")
print(f"  • Mean Policy Latency    : {best_metrics['mean_latency']:.2f} ms")
print("=" * 68)


   TEAM: Team_ReferenceSolution — OFFICIAL HACKATHON BENCHMARK SUBMISSION
  • Best Performing Agent  : Track 2 (RL Closed-Loop Agent)
  • Official Total Score   : 939.981 points
  • Alignment Success Rate : 100.0%
  • Mean Steps to Target   : 3.7 steps
  • Mean Residual Orbit RMS: 0.0101 mm
  • Mean Corrector Norm    : 3.85 A
  • Mean Policy Latency    : 0.41 ms
